# Signal Reconciliation — Redis DB1 (order_gateway) vs CSV (strategy_lab)

Đối chiếu nhanh signal `order_gateway` vừa publish lên Redis DB1 với signal tương ứng trong CSV export của `strategy_lab` (`runtime/exports/`), theo đúng `(strategy, symbol, timeframe, bartime)`.

Chỉ đọc — không ghi gì lên Redis/đĩa (HGETALL/LRANGE trên DB1, đọc CSV).

DB1 chắc chắn có ít signal hơn CSV (CSV là full-history, DB1 mới chạy vài lần test) — notebook chỉ so trên đoạn overlap (bartime có mặt ở cả 2 bên).

**Giới hạn đã biết**: HASH của `ma_cross` trên DB1 không có field entry (order MARKET, giá khớp thật lúc OF đặt lệnh chứ không cố định trước) — nên `ma_cross` chỉ đối chiếu được hướng signal (BUY/SELL) + atr; `combo` đối chiếu được cả entry price.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from order_gateway.src.candle_reader import create_client, normalize_app_config
from order_gateway.src.config import CONFIG

EXPORTS_DIR = REPO_ROOT / "runtime" / "exports"

config = normalize_app_config(CONFIG)
redis_config = config["redis"]
output_config = redis_config["output"]
db1 = create_client(redis_config, output_config["db"])
db1.ping()
print(f"Đã kết nối Redis DB{output_config['db']} (output) — key_prefix={output_config['key_prefix']!r}")

Đã kết nối Redis DB1 (output) — key_prefix='L_SIGNAL'


In [2]:
# Đọc toàn bộ signal hiện có trên DB1.
#
# ma_cross tự có dấu "_" trong tên nên khớp theo hậu tố chiến lược đã biết
# trước, không split "_" ngây thơ (cùng lý do với CLAUDE.md mục 7 —
# parse_signal_list_key trong research/redis_signal_audit.py cũ).
_KNOWN_STRATEGY_SUFFIXES = {"_COMBO": "combo", "_MA_CROSS": "ma_cross"}


def parse_signal_list_key(key: str, key_prefix: str) -> tuple[str, str, str] | None:
    """(symbol, timeframe, strategy) từ 1 key LIST signal, vd L_SIGNAL_US30_H4_COMBO."""
    prefix = f"{key_prefix}_"
    if not key.startswith(prefix):
        return None
    body = key[len(prefix):]
    for suffix, strategy in _KNOWN_STRATEGY_SUFFIXES.items():
        if body.endswith(suffix):
            symbol_tf = body[: -len(suffix)]
            if "_" not in symbol_tf:
                return None
            symbol, timeframe = symbol_tf.rsplit("_", 1)
            return symbol, timeframe, strategy
    return None


rows = []
for raw_key in db1.scan_iter(match=f"{output_config['key_prefix']}_*", count=1000, _type="list"):
    parsed = parse_signal_list_key(raw_key, output_config["key_prefix"])
    if parsed is None:
        continue
    symbol, timeframe, strategy = parsed
    for stamp in db1.lrange(raw_key, 0, -1):
        fields = db1.hgetall(f"{raw_key}:{stamp}")
        if not fields:
            continue
        rows.append({
            "strategy": strategy,
            "symbol": symbol,
            "timeframe": timeframe,
            "bartime": fields.get("bartime"),
            "side": fields.get("tradeSide"),
            "entry_db1": float(fields["stopPrice"]) if "stopPrice" in fields else None,
            "atr_db1": float(fields["atr"]) if "atr" in fields else None,
        })

df_db1 = pd.DataFrame(rows)
if not df_db1.empty:
    df_db1["bartime"] = pd.to_datetime(df_db1["bartime"], utc=True).dt.tz_localize(None)
    df_db1 = df_db1.sort_values(["strategy", "symbol", "timeframe", "bartime"]).reset_index(drop=True)

n_pairs = df_db1[["strategy", "symbol", "timeframe"]].drop_duplicates().shape[0] if not df_db1.empty else 0
print(f"DB1: {len(df_db1)} signal trên {n_pairs} pair (symbol, timeframe, strategy)")
df_db1

DB1: 248 signal trên 76 pair (symbol, timeframe, strategy)


,strategy,symbol,timeframe,bartime,side,entry_db1,atr_db1
0,combo,BTCUSD,H1,2026-09-19 20:00:00,SELL,80963.85,292.71
1,combo,BTCUSD,H1,2026-09-22 10:00:00,SELL,85661.60,551.75
2,combo,BTCUSD,H2,2026-09-20 01:00:00,SELL,80154.45,519.69
3,combo,DE40,H1,2026-09-22 06:00:00,SELL,25563.60,27.72
4,combo,DE40,H1,2026-09-22 09:00:00,BUY,25729.80,103.16
...,...,...,...,...,...,...,...
243,ma_cross,US500,M30,2026-09-22 23:00:00,BUY,NaN,4.74
244,ma_cross,US500,M30,2026-09-23 04:00:00,BUY,NaN,4.08
245,ma_cross,US500,M45,2026-09-18 16:30:00,SELL,NaN,12.85
246,ma_cross,US500,M45,2026-09-22 16:30:00,SELL,NaN,10.37


In [3]:
# Đọc CSV tương ứng trong runtime/exports/ cho đúng các pair đang có trên DB1.
# Chỉ đọc file mode "nt" (no_trend) — khớp og_config.yaml hiện tại
# (trend_filter_enabled=false cho cả combo/ma_cross).
_SIGNAL_SIDE = {1: "BUY", 2: "SELL"}


def read_export_csv(strategy: str, symbol: str, timeframe: str) -> pd.DataFrame:
    path = EXPORTS_DIR / f"{strategy}_{symbol}_{timeframe}_full_history_nt.csv"
    if not path.exists():
        return pd.DataFrame(columns=["bartime", "atr", "entry", "signal"])
    df = pd.read_csv(path, parse_dates=["bartime"])
    if "entry" not in df.columns:
        df["entry"] = None
    return df


csv_frames = []
if not df_db1.empty:
    for (strategy, symbol, timeframe), _ in df_db1.groupby(["strategy", "symbol", "timeframe"]):
        csv_df = read_export_csv(strategy, symbol, timeframe)
        csv_frames.append(csv_df.assign(strategy=strategy, symbol=symbol, timeframe=timeframe))

df_csv = pd.concat(csv_frames, ignore_index=True) if csv_frames else pd.DataFrame()
if not df_csv.empty:
    df_csv["side_csv"] = df_csv["signal"].map(_SIGNAL_SIDE)

print(f"CSV: đọc {len(csv_frames)} file, tổng {len(df_csv)} dòng signal cho các pair đang có trên DB1")

CSV: đọc 76 file, tổng 205746 dòng signal cho các pair đang có trên DB1


In [4]:
# Đối chiếu theo (strategy, symbol, timeframe, bartime).
if df_db1.empty:
    print("DB1 chưa có signal nào để đối chiếu.")
    comparison = pd.DataFrame()
else:
    comparison = df_db1.merge(
        df_csv,
        on=["strategy", "symbol", "timeframe", "bartime"],
        how="left",
    )
    comparison["found_in_csv"] = comparison["side_csv"].notna()
    comparison["side_match"] = comparison["side"] == comparison["side_csv"]
    comparison["entry_match"] = comparison.apply(
        lambda r: True if pd.isna(r["entry_db1"]) or pd.isna(r["entry"])
        else abs(r["entry_db1"] - r["entry"]) < 0.01,
        axis=1,
    )

display_cols = [
    "strategy", "symbol", "timeframe", "bartime",
    "side", "side_csv", "side_match",
    "entry_db1", "entry", "entry_match", "found_in_csv",
]
comparison[display_cols] if not comparison.empty else comparison

,strategy,symbol,timeframe,bartime,side,side_csv,side_match,entry_db1,entry,entry_match,found_in_csv
0,combo,BTCUSD,H1,2026-09-19 20:00:00,SELL,SELL,True,80963.85,80963.85,True,True
1,combo,BTCUSD,H1,2026-09-22 10:00:00,SELL,NaN,False,85661.60,NaN,True,False
2,combo,BTCUSD,H2,2026-09-20 01:00:00,SELL,SELL,True,80154.45,80154.45,True,True
3,combo,DE40,H1,2026-09-22 06:00:00,SELL,NaN,False,25563.60,NaN,True,False
4,combo,DE40,H1,2026-09-22 09:00:00,BUY,NaN,False,25729.80,NaN,True,False
...,...,...,...,...,...,...,...,...,...,...,...
243,ma_cross,US500,M30,2026-09-22 23:00:00,BUY,NaN,False,NaN,NaN,True,False
244,ma_cross,US500,M30,2026-09-23 04:00:00,BUY,NaN,False,NaN,NaN,True,False
245,ma_cross,US500,M45,2026-09-18 16:30:00,SELL,SELL,True,NaN,None,True,True
246,ma_cross,US500,M45,2026-09-22 16:30:00,SELL,NaN,False,NaN,NaN,True,False


In [5]:
# Tổng kết.
if comparison.empty:
    print("Không có gì để tổng kết.")
else:
    total = len(comparison)
    found = int(comparison["found_in_csv"].sum())
    side_ok = int((comparison["found_in_csv"] & comparison["side_match"]).sum())
    entry_ok = int((comparison["found_in_csv"] & comparison["entry_match"]).sum())

    print(f"Tổng {total} signal trên DB1.")
    print(f"  - Tìm thấy đúng bartime trong CSV: {found}/{total}")
    print(f"  - Khớp hướng BUY/SELL: {side_ok}/{found}")
    print(f"  - Khớp entry price (chỉ tính được khi cả 2 bên có entry, tức combo): {entry_ok}/{found}")

    mismatches = comparison[comparison["found_in_csv"] & (~comparison["side_match"] | ~comparison["entry_match"])]
    if not mismatches.empty:
        print("\nCÁC DÒNG LỆCH:")
        display(mismatches[display_cols])
    else:
        print("\nKhông có dòng nào lệch.")

    not_found = comparison[~comparison["found_in_csv"]]
    if not not_found.empty:
        print(f"\n{len(not_found)} signal DB1 KHÔNG tìm thấy bartime tương ứng trong CSV (CSV export chưa cập nhật tới bar này, hoặc lệch giờ đóng bar):")
        display(not_found[["strategy", "symbol", "timeframe", "bartime", "side"]])

Tổng 248 signal trên DB1.
  - Tìm thấy đúng bartime trong CSV: 98/248
  - Khớp hướng BUY/SELL: 98/98
  - Khớp entry price (chỉ tính được khi cả 2 bên có entry, tức combo): 98/98

Không có dòng nào lệch.

150 signal DB1 KHÔNG tìm thấy bartime tương ứng trong CSV (CSV export chưa cập nhật tới bar này, hoặc lệch giờ đóng bar):


,strategy,symbol,timeframe,bartime,side
1,combo,BTCUSD,H1,2026-09-22 10:00:00,SELL
3,combo,DE40,H1,2026-09-22 06:00:00,SELL
4,combo,DE40,H1,2026-09-22 09:00:00,BUY
5,combo,DE40,H1,2026-09-22 14:00:00,SELL
12,combo,FR40,H1,2026-09-22 07:00:00,SELL
...,...,...,...,...,...
242,ma_cross,US500,M30,2026-09-22 19:30:00,SELL
243,ma_cross,US500,M30,2026-09-22 23:00:00,BUY
244,ma_cross,US500,M30,2026-09-23 04:00:00,BUY
246,ma_cross,US500,M45,2026-09-22 16:30:00,SELL
